## Step 0 — Setup and Load

### Why This Step Exists

Before writing a single line of model code we need three things
in place:

1. All libraries imported in one place
   If a library is missing the notebook fails immediately here
   rather than halfway through a 10-minute training run.

2. All constants defined in one place
   PURGE_DAYS, N_SPLITS, RANDOM_STATE are used in multiple cells.
   Defining them once at the top means changing a value in one
   place updates the entire notebook automatically.
   No hunting through 20 cells to find where 5 was hardcoded.

3. Random seeds fixed before anything else runs
   XGBoost makes random decisions internally during training.
   Without a fixed seed, two runs of this notebook on identical
   data can produce different models and different results.
   Reproducibility is non-negotiable in a production pipeline.
   We fix numpy seed AND pass random_state to every model.

### Constants Defined Here

| Constant | Value | Reason |
|----------|-------|--------|
| RANDOM_STATE | 42 | Any fixed integer works — 42 is convention |
| PURGE_DAYS | 5 | Must match prediction horizon from notebook 03 |
| N_SPLITS | 5 | Five CV folds is standard for this dataset size |
| N_CLASSES | 3 | Sell=0, Hold=1, Buy=2 |

### Directory Setup

The models directory is created here if it does not exist yet.
All saved models, OOF arrays, and scalers go into ../models/
This keeps the project structure clean and predictable.

In [1]:
# ============================================================
# CELL 1 - Imports and Configuration
# ============================================================
# Import all libraries needed for this notebook upfront
# Fix all random seeds before any library initialises
# Define all constants in one place for easy modification
# Create output directory for saved models
# ============================================================

import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection     import TimeSeriesSplit
from sklearn.metrics             import (classification_report,
                                         balanced_accuracy_score,
                                         confusion_matrix,
                                         f1_score)
from sklearn.utils.class_weight  import compute_sample_weight

# Fix random seed for full reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Project constants — change here to update entire notebook
PURGE_DAYS = 5   # must match prediction horizon in notebook 03
N_SPLITS   = 5   # number of walk-forward CV folds
N_CLASSES  = 3   # Sell=0, Hold=1, Buy=2

# Class label map — used in print statements throughout
LABELS = {0: 'Sell', 1: 'Hold', 2: 'Buy'}

# File paths
PROCESSED_PATH = '../data/processed/all_stocks_features.csv'
MODELS_DIR     = '../models/'
os.makedirs(MODELS_DIR, exist_ok=True)

print('Environment ready')
print('=' * 45)
print(f'  XGBoost version  : {xgb.__version__}')
print(f'  Random state     : {RANDOM_STATE}')
print(f'  Purge days       : {PURGE_DAYS}')
print(f'  CV folds         : {N_SPLITS}')
print(f'  Classes          : {LABELS}')
print(f'  Models directory : {MODELS_DIR}')

Environment ready
  XGBoost version  : 3.2.0
  Random state     : 42
  Purge days       : 5
  CV folds         : 5
  Classes          : {0: 'Sell', 1: 'Hold', 2: 'Buy'}
  Models directory : ../models/


------------------------------------

### Cell 2 — Load and Verify Processed Data

#### Why We Reload and Re-verify Here

Notebook 03 saved the processed file and verified it at that time.
But notebooks are often run days or weeks apart.
Someone may have accidentally modified the file in between.
We never assume the file is clean — we verify it fresh every time.

#### What We Check

| Check | Expected | Why |
|-------|----------|-----|
| Shape rows | 29100 | Any deviation means file was modified |
| NaNs | 0 | A single NaN would corrupt model training silently |
| Tickers | 20 | Missing ticker = missing market sector |
| Date range | 2019-03-14 to 2024-12-20 | Confirms no rows were accidentally dropped |

#### Why We Sort Again

The file was sorted when saved in notebook 03.
But reading a CSV does not guarantee row order.
We sort again explicitly — this costs almost nothing
and guarantees all downstream operations see rows
in the correct chronological order per ticker.

#### Why We Drop macd and macd_signal Here

The processed CSV contains all 27 columns including macd
and macd_signal. We saved them in notebook 03 but decided
not to use them as model features.

They are dropped explicitly here for two reasons:

1. Clarity — anyone reading this notebook sees immediately
   that these columns are intentionally excluded.
   There is no ambiguity about whether they were forgotten.

2. Safety — dropping them now makes it impossible to
   accidentally include them in FEATURE_COLS later.
   Implicit exclusion is a future bug waiting to happen.

macd_hist is the only MACD-based feature we keep.
It is the final signal that traders actually act on.
macd and macd_signal are the intermediate ingredients
that produce macd_hist — keeping them would give the model
redundant information it has to untangle unnecessarily.

After the drop the dataframe has 25 columns:
7 original (Date, OHLCV, Ticker) + 17 features + 1 target.

#### Assert-First Philosophy

Every check is an assert not just a print.
A print warns you. An assert stops execution immediately.
Silent corruption is the most dangerous failure mode
in a multi-notebook pipeline — loud failures are far safer.

In [2]:
# ============================================================
# CELL 2 - Load and Verify Processed Data
# ============================================================
# Load the feature set produced by notebook 03
# Sort by Ticker then Date — never assume CSV row order
# Run hard assertions on shape, NaNs, tickers, date range
# Drop macd and macd_signal — redundant, macd_hist is kept
# Any assertion failure means notebook 03 output is corrupted
# Fix the issue in notebook 03 before continuing here
# ============================================================

df = pd.read_csv(PROCESSED_PATH, parse_dates=['Date'])
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print('Processed data loaded')
print('=' * 50)
print(f'  Shape      : {df.shape}')
print(f'  NaNs       : {df.isnull().sum().sum()}')
print(f'  Tickers    : {df["Ticker"].nunique()}')
print(f'  Date range : {df["Date"].min().date()} '
      f'to {df["Date"].max().date()}')
print(f'  Columns    : {list(df.columns)}')

# Hard assertions — fail immediately if anything is wrong
assert df.shape[0]             == 29100, \
    f'Expected 29100 rows, got {df.shape[0]} — check notebook 03'
assert df.shape[1]             == 27, \
    f'Expected 27 columns, got {df.shape[1]} — check notebook 03'
assert df.isnull().sum().sum() == 0, \
    'NaNs found in processed data — check notebook 03'
assert df['Ticker'].nunique()  == 20, \
    f'Expected 20 tickers, got {df["Ticker"].nunique()}'

print('\nAll load assertions passed ✅')

# ----------------------------------------------------------
# Drop macd and macd_signal
# macd_hist is the only MACD feature we use
# macd and macd_signal are intermediate ingredients that
# produce macd_hist — keeping them is redundant information
# Dropping here makes accidental use impossible downstream
# ----------------------------------------------------------
df = df.drop(columns=['macd', 'macd_signal'])

print(f'\nDropped : macd, macd_signal')
print(f'Shape after drop : {df.shape}')

assert df.shape[1]          == 25, \
    f'Expected 25 columns after drop, got {df.shape[1]}'
assert 'macd'               not in df.columns, \
    'macd column still present — drop failed'
assert 'macd_signal'        not in df.columns, \
    'macd_signal column still present — drop failed'
assert 'macd_hist'          in df.columns, \
    'macd_hist column missing — incorrectly dropped'

print('Column drop verified ✅')
print()
print('Final columns:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:>2}. {col}')

Processed data loaded
  Shape      : (29100, 27)
  NaNs       : 0
  Tickers    : 20
  Date range : 2019-03-14 to 2024-12-20
  Columns    : ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'price_vs_ma20', 'price_vs_ma50', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'bb_width', 'atr_14', 'volume_ratio', 'volume_trend', 'price_volume_signal', 'market_return_1d', 'day_of_week', 'month', 'quarter', 'target']

All load assertions passed ✅

Dropped : macd, macd_signal
Shape after drop : (29100, 25)
Column drop verified ✅

Final columns:
   1. Date
   2. Close
   3. High
   4. Low
   5. Open
   6. Volume
   7. Ticker
   8. return_1d
   9. return_5d
  10. return_10d
  11. return_20d
  12. price_vs_ma20
  13. price_vs_ma50
  14. rsi_14
  15. macd_hist
  16. bb_width
  17. atr_14
  18. volume_ratio
  19. volume_trend
  20. price_volume_signal
  21. market_return_1d
  22. day_of_week
  23. month
  24. quarter
  25. target


-------------------------------

### Cell 3 — Train / Validation Split by Date

#### Why We Split by Date and Not Randomly

This is a time series problem. The order of data matters.

If we split randomly:
  Training data includes rows from 2023
  Validation data includes rows from 2021
  The model trains on the future and tests on the past
  This is called look-ahead bias
  Results will look excellent but are completely fake
  The model will fail badly on real live data

If we split by date:
  Training data   →  everything up to Dec 31 2023
  Validation data →  everything from Jan 01 2024 onwards
  The model only ever sees past data during training
  Validation tests on data the model has never seen
  This reflects real-world trading conditions honestly

#### Our Split Boundary

  Train      :  2019-03-14  to  2023-12-31
  
  Validation :  2024-01-01  to  2024-12-20

#### Why This Split Makes Sense for Our Timeline

Our project has three distinct time periods:

  Training period   :  2019-2023  →  model learns from this
  Validation period :  2024       →  model tested on this
  Live prediction   :  2025       →  unseen, downloaded live
  Dashboard live    :  2026       →  real-time predictions

The model trains on 5 distinct market regimes:
  2019  →  normal bull market
  2020  →  COVID crash and recovery
  2021  →  post-COVID bull run
  2022  →  rate hike bear market
  2023  →  recovery and stabilisation

Then validates on 2024 — a fresh bull market it has never seen.

2025 data is never part of this notebook at all.
It is downloaded live from yfinance in the dashboard.
The model has never seen a single row from 2025.
This is the most honest test of real-world performance.

2026 is when the dashboard runs in production.
Every prediction made in 2026 is completely live —
the model trained in 2023, validated in 2024,
tested conceptually on 2025, now predicting 2026.
This is a true out-of-sample deployment.

#### The Vault Rule

After this cell the validation set is locked.
It will not be touched again until Step 7 — final evaluation.

  No hyperparameter tuning on validation data
  No threshold adjustments based on validation data
  No feature selection based on validation data
  No peeking at validation results during training

Every decision from here until Step 7 uses only training data.
This is the only way to get an honest performance number.

#### What We Verify

  No date overlap between train and validation
  All 20 tickers present in both sets
  Train is strictly before validation chronologically
  Row counts add up to total dataset size

In [3]:
# ============================================================
# CELL 3 - Train / Validation Split by Date
# ============================================================
TRAIN_END = '2023-12-31'
VAL_START = '2024-01-01'

df_train = df[df['Date'] <= TRAIN_END].copy()
df_val   = df[df['Date'] >= VAL_START].copy()

# Reset index — critical for OOF array alignment in Cell 7
# Without this df_train indices are from the original df
# and will be out of bounds for oof_probs array
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)

print('Train / Validation split complete')
print('=' * 55)
print(f'  Train      : {df_train["Date"].min().date()} '
      f'to {df_train["Date"].max().date()} '
      f'| {len(df_train):,} rows '
      f'({len(df_train)/len(df)*100:.1f}%)')
print(f'  Validation : {df_val["Date"].min().date()} '
      f'to {df_val["Date"].max().date()} '
      f'| {len(df_val):,} rows '
      f'({len(df_val)/len(df)*100:.1f}%)')
print(f'  Total      : {len(df_train) + len(df_val):,} rows')
print()
print('  Live 2025  : downloaded from yfinance in dashboard')
print('  Live 2026  : real-time predictions in production')

# Verify no date overlap
assert df_train['Date'].max() < df_val['Date'].min(), \
    'DATE OVERLAP DETECTED — train and val overlap — LEAKAGE'

# Verify all 20 tickers in both sets
assert df_train['Ticker'].nunique() == 20, \
    f'Train missing tickers — only {df_train["Ticker"].nunique()} found'
assert df_val['Ticker'].nunique() == 20, \
    f'Val missing tickers — only {df_val["Ticker"].nunique()} found'

# Verify row counts add up
assert len(df_train) + len(df_val) == len(df), \
    f'Row count mismatch — {len(df_train)} + {len(df_val)} != {len(df)}'

# Verify index is clean after reset
assert df_train.index[0]  == 0, \
    'df_train index does not start at 0 — reset failed'
assert df_train.index[-1] == len(df_train) - 1, \
    'df_train index is not contiguous — reset failed'

print()
print('Split assertions passed ✅')
print('Index reset verified ✅')
print()

# Class distribution comparison
print('Class distribution comparison:')
print(f'  {"Class":<8} {"Train %":>8} {"Val %":>8} {"Difference":>12}')
print(f'  {"-" * 40}')
for cls in range(N_CLASSES):
    train_pct = (df_train['target'] == cls).mean() * 100
    val_pct   = (df_val['target']   == cls).mean() * 100
    diff      = val_pct - train_pct
    label     = LABELS[cls]
    flag      = '⚠️' if abs(diff) > 5 else '✅'
    print(f'  {label:<8} {train_pct:>8.1f}% {val_pct:>8.1f}% '
          f'{diff:>+11.1f}%  {flag}')

print()
print('Validation set is now LOCKED until Step 7 ✅')

Train / Validation split complete
  Train      : 2019-03-14 to 2023-12-29 | 24,180 rows (83.1%)
  Validation : 2024-01-02 to 2024-12-20 | 4,920 rows (16.9%)
  Total      : 29,100 rows

  Live 2025  : downloaded from yfinance in dashboard
  Live 2026  : real-time predictions in production

Split assertions passed ✅
Index reset verified ✅

Class distribution comparison:
  Class     Train %    Val %   Difference
  ----------------------------------------
  Sell         24.9%     20.7%        -4.2%  ✅
  Hold         42.5%     45.9%        +3.5%  ✅
  Buy          32.6%     33.4%        +0.8%  ✅

Validation set is now LOCKED until Step 7 ✅


-------------------------------------------

### Cell 4 — Define Feature Columns

#### Why We Define Features Explicitly in One Place

Feature columns are referenced in multiple places:
  Cell 7  →  OOF training loop
  Cell 9  →  final model training
  Cell 10 →  validation evaluation
  Cell 11 →  feature importance
  Cell 12 →  saving to disk for notebook 07

If we hardcode the list in each cell separately,
one typo in one cell creates a silent mismatch.
The model trains on 17 features but evaluates on 16.
No error is raised — results are just subtly wrong.

Defining once here and referencing everywhere else
means a single change updates the entire notebook.

#### Why 17 Features and Not All 25 Columns

The dataframe has 25 columns after the drop in Cell 2.
We only use 17 of them as model input.

Excluded columns and why:

  Date       →  not a feature, used for splitting only
  Open       →  redundant, Close captures price level
  High       →  redundant, ATR already captures the range
  Low        →  redundant, ATR already captures the range
  Close      →  raw price, not comparable across stocks
  Volume     →  raw volume, volume_ratio normalises this
  Ticker     →  categorical identifier, not a signal
  target     →  this is what we predict, never an input

#### The 17 Features and Their Groups

Price group — 6 features
  Captures momentum and trend at different time horizons
  Returns are scale-free — +2% means the same for any stock

Technical group — 4 features
  RSI captures momentum exhaustion
  MACD histogram captures trend direction changes
  Bollinger width captures volatility regime
  ATR captures normalised daily swing size

Volume group — 3 features
  Volume ratio captures unusual activity
  Volume trend captures growing or shrinking interest
  Price volume signal combines direction with conviction

Market group — 1 feature
  Captures broad market sentiment on each day
  Tells the model whether the whole market is up or down
  No single stock indicator can provide this context

Time group — 3 features
  Day of week captures Monday effect
  Month captures January effect and earnings seasons
  Quarter captures quarterly earnings cycles

#### Why This List Is Saved to Disk

In notebook 07 the meta-model loads predictions from
all three base models. It must use the exact same
feature list to generate those predictions.

If notebook 04 used 17 features but notebook 07
accidentally uses 16, the models produce incompatible
outputs and the stacking ensemble breaks.

Saving the feature list as a file and loading it in
every downstream notebook guarantees consistency.

In [4]:
# ============================================================
# CELL 4 - Define Feature Columns
# ============================================================
# Explicitly list all 17 features used for model training
# Group them by type for clarity and documentation
# Verify every feature exists in the dataframe
# Save list to disk — notebooks 05 06 07 load this file
# to guarantee identical feature sets across all models
# ============================================================

FEATURE_COLS = [
    # --- Price features (6) ---
    # Scale-free returns at multiple horizons
    # Comparable across all stocks regardless of price level
    'return_1d',
    'return_5d',
    'return_10d',
    'return_20d',
    'price_vs_ma20',
    'price_vs_ma50',

    # --- Technical indicators (4) ---
    # Industry standard signals used by traders worldwide
    'rsi_14',           # momentum — overbought/oversold
    'macd_hist',        # trend direction change signal
    'bb_width',         # volatility regime
    'atr_14',           # normalised daily swing size

    # --- Volume features (3) ---
    # Volume confirms or contradicts price moves
    'volume_ratio',          # unusual activity vs 20d avg
    'volume_trend',          # growing or shrinking interest
    'price_volume_signal',   # direction × conviction

    # --- Market-wide feature (1) ---
    # Average return across all 20 stocks on each date
    # Captures macro events no single stock indicator can see
    'market_return_1d',

    # --- Time features (3) ---
    # Calendar effects documented in financial research
    'day_of_week',   # Monday effect
    'month',         # January effect, earnings seasons
    'quarter',       # quarterly earnings cycles
]

TARGET_COL = 'target'

# ── Verification ──────────────────────────────────────────
# Confirm every feature exists in the dataframe
missing = [c for c in FEATURE_COLS if c not in df.columns]
assert len(missing) == 0, \
    f'Missing features in dataframe: {missing}'

# Confirm target exists
assert TARGET_COL in df.columns, \
    f'Target column {TARGET_COL} not found in dataframe'

# Confirm no overlap between features and target
assert TARGET_COL not in FEATURE_COLS, \
    'Target column is inside FEATURE_COLS — model would cheat'

# Confirm excluded columns are not accidentally included
excluded = ['Date', 'Open', 'High', 'Low',
            'Close', 'Volume', 'Ticker']
accidental = [c for c in excluded if c in FEATURE_COLS]
assert len(accidental) == 0, \
    f'Raw columns accidentally in FEATURE_COLS: {accidental}'

# ── Summary ───────────────────────────────────────────────
print('Feature columns defined and verified')
print('=' * 50)
print(f'  Total features : {len(FEATURE_COLS)}')
print(f'  Target column  : {TARGET_COL}')
print()

groups = {
    'Price      (6)' : FEATURE_COLS[0:6],
    'Technical  (4)' : FEATURE_COLS[6:10],
    'Volume     (3)' : FEATURE_COLS[10:13],
    'Market     (1)' : FEATURE_COLS[13:14],
    'Time       (3)' : FEATURE_COLS[14:17],
}
for group, cols in groups.items():
    print(f'  {group} : {cols}')

print()

# ── Save to disk ───────────────────────────────────────────
# All downstream notebooks load this file
# Guarantees identical feature sets across all 3 base models
joblib.dump(FEATURE_COLS, f'{MODELS_DIR}feature_cols.pkl')
print(f'  Feature list saved to : {MODELS_DIR}feature_cols.pkl')

# Verify saved file loads correctly
loaded_cols = joblib.load(f'{MODELS_DIR}feature_cols.pkl')
assert loaded_cols == FEATURE_COLS, \
    'Saved feature list does not match — joblib save failed'
print(f'  Feature list verified  : reload matches original ✅')
print()
print('All feature assertions passed ✅')

Feature columns defined and verified
  Total features : 17
  Target column  : target

  Price      (6) : ['return_1d', 'return_5d', 'return_10d', 'return_20d', 'price_vs_ma20', 'price_vs_ma50']
  Technical  (4) : ['rsi_14', 'macd_hist', 'bb_width', 'atr_14']
  Volume     (3) : ['volume_ratio', 'volume_trend', 'price_volume_signal']
  Market     (1) : ['market_return_1d']
  Time       (3) : ['day_of_week', 'month', 'quarter']

  Feature list saved to : ../models/feature_cols.pkl
  Feature list verified  : reload matches original ✅

All feature assertions passed ✅


-------------------------

### Cell 5 — Compute Class Weights

#### The Problem — Class Imbalance

Our training set has this distribution:
  Sell (0) :  24.9%
  Hold (1) :  42.5%   ← nearly double Sell
  Buy  (2) :  32.6%

Without correction XGBoost will be biased toward Hold.
Here is why:

  Imagine the model is lazy and always predicts Hold.
  It would be correct 42.5% of the time.
  That sounds reasonable but it is completely useless —
  it never predicts a single Buy or Sell signal.
  A trader using this model would never make a trade.

The model needs to be told:
  Pay more attention to Sell rows — they are rare
  Pay less attention to Hold rows — they dominate

#### The Solution — Sample Weights

We assign a weight to every row in the training set.
Rows from rare classes get higher weights.
Rows from common classes get lower weights.

The formula is:
  weight = total_rows / (n_classes × class_count)

Example with our data:
  Sell weight = 24180 / (3 × 6024)  = 1.34  ← paid more attention
  Hold weight = 24180 / (3 × 10268) = 0.79  ← paid less attention
  Buy  weight = 24180 / (3 × 7888)  = 1.02  ← roughly neutral

XGBoost uses these weights during training.
A misclassified Sell row causes more penalty
than a misclassified Hold row.
This forces the model to learn all three classes properly.

#### Why sample_weight and Not class_weight

XGBoost does not have a class_weight parameter
like sklearn's LogisticRegression does.
Instead it accepts sample_weight — one weight per row.

sklearn's compute_sample_weight('balanced') handles this
automatically and correctly for any number of classes.
We use it here rather than computing manually
to avoid arithmetic errors.

#### Important — Weights Are for Training Only

Sample weights are never applied to:
  Validation set evaluation
  OOF prediction generation
  Feature importance calculation

They are only passed to model.fit() during training.
Evaluation always uses raw unweighted predictions
so metrics reflect true real-world performance.

In [5]:
# ============================================================
# CELL 5 - Compute Class Weights
# ============================================================
# Compute sample weights for training set only
# Inversely proportional to class frequency
# Rare classes get higher weight so model learns them
# Weights passed to model.fit() only — not to evaluation
# ============================================================

y_train_full = df_train[TARGET_COL].values.astype(int)

sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=y_train_full
)

print('Class weights computed')
print('=' * 55)
print(f'  {"Class":<8} {"Rows":>8} {"Pct":>8} '
      f'{"Weight":>10} {"Meaning"}')
print(f'  {"-" * 50}')

unique_classes, counts = np.unique(y_train_full,
                                    return_counts=True)
for cls, count in zip(unique_classes, counts):
    pct    = count / len(y_train_full) * 100
    weight = sample_weights[y_train_full == cls][0]
    label  = LABELS[cls]
    if weight > 1.0:
        meaning = 'more attention'
    elif weight < 1.0:
        meaning = 'less attention'
    else:
        meaning = 'neutral'
    print(f'  {label:<8} {count:>8,} {pct:>8.1f}% '
          f'{weight:>10.4f}  {meaning}')

print()
print(f'  Total rows           : {len(sample_weights):,}')
print(f'  Weight array shape   : {sample_weights.shape}')
print(f'  Min weight           : {sample_weights.min():.4f}')
print(f'  Max weight           : {sample_weights.max():.4f}')
print()

# Verify weights are positive — negative weights crash XGBoost
assert (sample_weights > 0).all(), \
    'Negative weights found — compute_sample_weight failed'

# Verify shape matches training set
assert len(sample_weights) == len(df_train), \
    f'Weight count {len(sample_weights)} != ' \
    f'train rows {len(df_train)}'

print('All weight assertions passed ✅')
print()
print('Note: weights are passed to model.fit() only')
print('      never applied to validation or evaluation')

Class weights computed
  Class        Rows      Pct     Weight Meaning
  --------------------------------------------------
  Sell        6,025     24.9%     1.3378  more attention
  Hold       10,270     42.5%     0.7848  less attention
  Buy         7,885     32.6%     1.0222  more attention

  Total rows           : 24,180
  Weight array shape   : (24180,)
  Min weight           : 0.7848
  Max weight           : 1.3378

All weight assertions passed ✅

Note: weights are passed to model.fit() only
      never applied to validation or evaluation


-------------------

### Cell 6 — XGBoost Baseline Parameters

#### Why We Define Parameters Separately

Parameters are defined in one dictionary before the
training loop for the same reason features are defined
in one list — change once, update everywhere.

The same parameter dictionary is used in:
  Cell 7  →  each fold of the OOF loop
  Cell 9  →  final model training on full train set

If we hardcoded parameters inside each cell separately,
changing one value would require finding and updating
every cell that uses it. That is a maintenance bug
waiting to happen.

#### Why Baseline First — No Grid Search Yet

We are building a baseline model in this notebook.
A baseline answers one question:
  Does XGBoost produce a reasonable signal at all?

If the baseline F1 score is near 0.33 (random chance
for a 3-class problem) then the features are not
predictive and we have a bigger problem to solve.

If the baseline is above 0.40 we have a working signal
and tuning will improve it further in notebook 07.

Tuning before establishing a baseline is premature.
You cannot know if tuning helped without knowing
where you started.

#### Parameter Explanations

objective = multi:softprob
  Tells XGBoost this is a multi-class classification task.
  softprob means output probabilities for each class
  not just the winning class label.
  We need probabilities for the stacking meta-model.

num_class = 3
  Three output classes: Sell=0, Hold=1, Buy=2
  XGBoost needs to know this upfront to build
  the correct output layer.

n_estimators = 500
  Number of trees to build.
  More trees = more learning but slower training.
  500 is a strong baseline — not too few, not excessive.
  We use early stopping in the OOF loop to prevent
  the model from overtraining on any fold.

learning_rate = 0.05
  How much each tree corrects the previous tree's errors.
  Lower = slower learning = more trees needed = less overfit
  0.05 with 500 trees is a well-balanced combination.
  Industry standard starting point.

max_depth = 6
  Maximum depth of each decision tree.
  Deeper = more complex patterns = more overfit risk.
  6 is the XGBoost default and works well for most
  financial tabular datasets.

subsample = 0.8
  Each tree only sees 80% of training rows randomly.
  This prevents any single tree from memorising
  the training data. Adds robustness.

colsample_bytree = 0.8
  Each tree only sees 80% of features randomly.
  Forces trees to learn from different feature combinations.
  Prevents over-reliance on any single feature.

eval_metric = mlogloss
  Multiclass log loss — standard metric for probability
  calibration in multi-class problems.
  Lower is better. Used internally by XGBoost
  to monitor training progress.

n_jobs = -1
  Use all available CPU cores for training.
  Makes no difference to results — only to speed.

In [6]:
# ============================================================
# CELL 6 - XGBoost Baseline Parameters
# ============================================================
# Define all parameters in one dictionary
# Same dict used in Cell 7 (OOF loop) and Cell 9 (final model)
# Baseline parameters — no grid search at this stage
# Tuning happens in notebook 07 after baseline is established
# ============================================================

XGB_PARAMS = {
    # Task definition
    'objective'        : 'multi:softprob',  # probability output
    'num_class'        : N_CLASSES,          # Sell, Hold, Buy

    # Tree structure
    'n_estimators'     : 500,
    'max_depth'        : 6,
    'learning_rate'    : 0.05,

    # Randomisation — prevents overfitting
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,

    # Evaluation and reproducibility
    'eval_metric'      : 'mlogloss',
    'random_state'     : RANDOM_STATE,
    'n_jobs'           : -1,
    'verbosity'        : 0
}

print('XGBoost baseline parameters defined')
print('=' * 50)
print(f'  {"Parameter":<22} {"Value"}')
print(f'  {"-" * 40}')
for param, value in XGB_PARAMS.items():
    print(f'  {param:<22} {value}')

print()
print('These parameters are used in:')
print('  Cell 7  →  OOF cross-validation loop')
print('  Cell 9  →  Final model on full train set')
print()
print('Tuning strategy:')
print('  Step 1  →  Run baseline (this notebook)')
print('  Step 2  →  Check baseline F1 score')
print('  Step 3  →  Tune in notebook 07 if needed')
print()

# Sanity check — confirm key parameters are set correctly
assert XGB_PARAMS['num_class']  == N_CLASSES, \
    'num_class must match N_CLASSES constant'
assert XGB_PARAMS['objective']  == 'multi:softprob', \
    'objective must be multi:softprob for probability output'
assert XGB_PARAMS['random_state'] == RANDOM_STATE, \
    'random_state must match global RANDOM_STATE'

print('Parameter assertions passed ✅')

XGBoost baseline parameters defined
  Parameter              Value
  ----------------------------------------
  objective              multi:softprob
  num_class              3
  n_estimators           500
  max_depth              6
  learning_rate          0.05
  subsample              0.8
  colsample_bytree       0.8
  eval_metric            mlogloss
  random_state           42
  n_jobs                 -1
  verbosity              0

These parameters are used in:
  Cell 7  →  OOF cross-validation loop
  Cell 9  →  Final model on full train set

Tuning strategy:
  Step 1  →  Run baseline (this notebook)
  Step 2  →  Check baseline F1 score
  Step 3  →  Tune in notebook 07 if needed

Parameter assertions passed ✅


-------------------

### Cell 7 — Walk-Forward Out-of-Fold Cross Validation

#### What This Cell Does

This cell trains XGBoost 5 times — once per fold.
Each fold trains on a different slice of the training set
and predicts on a slice it has never seen.

The predictions collected across all 5 folds form the
Out-of-Fold (OOF) predictions — a complete set of honest
predictions covering the entire training set where every
single row was predicted by a model that never trained on it.

These OOF predictions are saved and used in notebook 07
to train the Logistic Regression meta-model.

#### Why OOF and Not Just Validation Predictions

The meta-model needs training data.
The only data available for meta-model training
is the training set itself.

If we generated predictions on the training set using
a model that was trained on the training set:
  The model has already memorised these rows
  Its predictions will be overconfident and unrealistic
  The meta-model trains on fake perfect predictions
  Real performance will be much worse

OOF predictions solve this:
  Every row is predicted by a model that never saw it
  Predictions are honest — same difficulty as real data
  Meta-model trains on realistic signal strengths
  Stacking ensemble works correctly

#### Four Rules Applied in This Cell

Rule 1 — Purge Gap
  The last 5 dates of each training fold are dropped
  before fitting the model.
  This prevents target leakage at fold boundaries.
  Our target looks 5 days forward — without purging,
  the last 5 training rows peek into validation territory.

Rule 2 — Date-Based Splitting
  We split on unique dates not on row indices.
  sklearn TimeSeriesSplit on row indices would cross
  ticker boundaries — ADBE 2019 data would appear
  after AAPL 2024 data which is chronologically wrong.
  Splitting on dates ensures all 20 tickers are
  correctly represented in every fold.

Rule 3 — Assert After Every Fold
  After creating each fold we assert:
    No date overlap between train and validation
    All 20 tickers present in training fold
    All 20 tickers present in validation fold
  A failing assert stops execution immediately.
  Silent corruption is far more dangerous than
  a loud failure that stops the notebook.

Rule 4 — OOF Coverage Tracking
  A separate array tracks how many times each row
  was predicted. After the loop every predicted row
  must have coverage exactly equal to 1.
  Zero means a row was never predicted — gap in OOF.
  Two means a row was predicted twice — corruption.
  Both are caught in Cell 8 sanity validation.

#### What the Fold Scores Tell Us

Each fold reports F1 macro and Balanced Accuracy.
F1 macro treats all three classes equally.
Balanced Accuracy adjusts for class imbalance.

A good baseline for a 3-class financial problem:
  F1 macro > 0.35   →  model is learning signal
  F1 macro > 0.45   →  strong baseline
  F1 macro < 0.33   →  near random — features may be weak

We expect scores around 0.38 to 0.48 for this dataset.
Financial data is noisy — do not expect 0.90+ scores.
A model claiming 0.90 accuracy on stock data is
almost certainly overfitting or has data leakage.

#### Fold Structure With 5 Splits

TimeSeriesSplit always grows the training window:

  Fold 1  Train: ~4,800 rows   Val: ~4,800 rows
  Fold 2  Train: ~9,600 rows   Val: ~4,800 rows
  Fold 3  Train: ~14,400 rows  Val: ~4,800 rows
  Fold 4  Train: ~19,200 rows  Val: ~4,800 rows
  Fold 5  Train: ~20,000 rows  Val: ~4,800 rows

Each fold trains on strictly past dates.
Each fold validates on strictly future dates.
The purge removes 5 dates at every boundary.

In [7]:
# ============================================================
# CELL 7 - Walk-Forward OOF Cross-Validation
# ============================================================
# Rule 1 : purge last 5 dates from each training fold
# Rule 2 : split on unique dates not row indices
# Rule 3 : assert checks after every fold split
# Rule 4 : track OOF coverage — catch gaps and duplicates
# Fix    : use positional index after df_train.reset_index()
#          fold_val.index was referencing original df index
#          causing out of bounds on oof_probs array
# ============================================================

# Get sorted unique dates from training set only
# Never include validation dates in this array
unique_dates = np.array(sorted(df_train['Date'].unique()))

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

# Initialise OOF arrays
# oof_probs    : stores 3-class probabilities per row
# oof_coverage : tracks how many times each row is predicted
#                must be exactly 0 or 1 after loop completes
oof_probs    = np.full((len(df_train), N_CLASSES), np.nan)
oof_coverage = np.zeros(len(df_train), dtype=int)

fold_scores = []

print('Walk-Forward OOF Cross-Validation')
print('=' * 65)
print(f'  Total unique dates in train : {len(unique_dates)}')
print(f'  Number of folds             : {N_SPLITS}')
print(f'  Purge days per boundary     : {PURGE_DAYS}')
print('=' * 65)

for fold, (train_idx, val_idx) in enumerate(
        tscv.split(unique_dates)):

    # ── Rule 2 : date-based splitting ─────────────────────
    train_dates = unique_dates[train_idx]
    val_dates   = unique_dates[val_idx]

    # ── Rule 1 : purge last 5 dates from training fold ────
    train_dates_purged = train_dates[:-PURGE_DAYS]

    # Filter dataframe by purged dates
    fold_train = df_train[
        df_train['Date'].isin(train_dates_purged)
    ]
    fold_val = df_train[
        df_train['Date'].isin(val_dates)
    ]

    # ── Rule 3 : assert checks ────────────────────────────
    assert fold_train['Date'].max() < fold_val['Date'].min(), \
        f'Fold {fold+1}: date overlap — LEAKAGE DETECTED'
    assert fold_train['Ticker'].nunique() == 20, \
        f'Fold {fold+1}: only ' \
        f'{fold_train["Ticker"].nunique()} tickers in train'
    assert fold_val['Ticker'].nunique() == 20, \
        f'Fold {fold+1}: only ' \
        f'{fold_val["Ticker"].nunique()} tickers in val'

    # ── Prepare arrays ────────────────────────────────────
    X_fold_train = fold_train[FEATURE_COLS].values
    y_fold_train = fold_train[TARGET_COL].values.astype(int)
    X_fold_val   = fold_val[FEATURE_COLS].values
    y_fold_val   = fold_val[TARGET_COL].values.astype(int)

    # Compute sample weights for this fold only
    fold_weights = compute_sample_weight(
        'balanced', y=y_fold_train
    )

    # ── Train XGBoost on this fold ────────────────────────
    model_fold = xgb.XGBClassifier(**XGB_PARAMS)
    model_fold.fit(
        X_fold_train, y_fold_train,
        sample_weight=fold_weights,
        verbose=False
    )

    # ── Generate OOF predictions ──────────────────────────
    fold_probs = model_fold.predict_proba(X_fold_val)

    # ── Store aligned to df_train positional index ────────
    # Rule 4 : use positional index from df_train
    # df_train was reset_index in Cell 3 so index runs
    # cleanly from 0 to 24179 matching oof_probs array size
    # fold_val.index references positions in df_train
    # which are now safe to use directly on oof_probs
    val_positions = df_train[
        df_train['Date'].isin(val_dates)
    ].index.tolist()

    oof_probs[val_positions]     = fold_probs
    oof_coverage[val_positions] += 1

    # ── Fold metrics ──────────────────────────────────────
    fold_preds = np.argmax(fold_probs, axis=1)
    fold_f1    = f1_score(
        y_fold_val, fold_preds, average='macro'
    )
    fold_bal   = balanced_accuracy_score(
        y_fold_val, fold_preds
    )
    fold_scores.append(fold_f1)

    print(f'\nFold {fold+1} / {N_SPLITS}')
    print(f'  Train : {fold_train["Date"].min().date()} '
          f'to {fold_train["Date"].max().date()} '
          f'| {len(fold_train):,} rows')
    print(f'  Val   : {fold_val["Date"].min().date()} '
          f'to {fold_val["Date"].max().date()} '
          f'| {len(fold_val):,} rows')
    print(f'  Purged: last {PURGE_DAYS} dates removed '
          f'from training boundary')
    print(f'  F1 Macro          : {fold_f1:.4f}')
    print(f'  Balanced Accuracy : {fold_bal:.4f}')

print()
print('=' * 65)
print('OOF Cross-Validation Complete')
print(f'  Mean F1 (macro)   : {np.mean(fold_scores):.4f}')
print(f'  Std  F1           : {np.std(fold_scores):.4f}')
print(f'  Min  F1           : {np.min(fold_scores):.4f}')
print(f'  Max  F1           : {np.max(fold_scores):.4f}')
print()
print('OOF arrays ready for Cell 8 sanity validation')
print(f'  oof_probs shape    : {oof_probs.shape}')
print(f'  oof_coverage shape : {oof_coverage.shape}')

Walk-Forward OOF Cross-Validation
  Total unique dates in train : 1209
  Number of folds             : 5
  Purge days per boundary     : 5

Fold 1 / 5
  Train : 2019-03-14 to 2019-12-24 | 3,980 rows
  Val   : 2020-01-03 to 2020-10-19 | 4,020 rows
  Purged: last 5 dates removed from training boundary
  F1 Macro          : 0.3609
  Balanced Accuracy : 0.3615

Fold 2 / 5
  Train : 2019-03-14 to 2020-10-12 | 8,000 rows
  Val   : 2020-10-20 to 2021-08-06 | 4,020 rows
  Purged: last 5 dates removed from training boundary
  F1 Macro          : 0.3554
  Balanced Accuracy : 0.3582

Fold 3 / 5
  Train : 2019-03-14 to 2021-07-30 | 12,020 rows
  Val   : 2021-08-09 to 2022-05-24 | 4,020 rows
  Purged: last 5 dates removed from training boundary
  F1 Macro          : 0.3679
  Balanced Accuracy : 0.3805

Fold 4 / 5
  Train : 2019-03-14 to 2022-05-17 | 16,040 rows
  Val   : 2022-05-25 to 2023-03-14 | 4,020 rows
  Purged: last 5 dates removed from training boundary
  F1 Macro          : 0.3607
  Balanc

------------------------------------

### Cell 8 — OOF Sanity Validation

#### Why This Cell Exists

Cell 7 ran 5 separate training loops and stored predictions
into the oof_probs array. If anything went wrong silently
during that loop — a fold that stored to wrong positions,
a probability array that did not sum to 1, a row that was
predicted twice or never predicted at all — the corruption
flows directly into notebook 07 and poisons the meta-model.

There is no error message for this kind of corruption.
The code runs. The numbers look plausible. The meta-model
trains. The ensemble produces predictions. Everything appears
to work — but the stacking layer is built on a corrupted
foundation and the final model is unreliable.

This cell is the firewall. Six checks are run. Every single
one must pass before we proceed to Cell 9. A single failure
stops execution immediately with a clear message telling you
exactly what went wrong and where to look.

#### The Six Checks

Check 1 — No row predicted twice
  oof_coverage must never exceed 1 for any row.
  If a row appears in two different fold validation sets
  it means our date-based splitting has an overlap bug.
  The second prediction silently overwrites the first.
  The meta-model trains on corrupted probabilities.

Check 2 — Coverage count is sensible
  The first fold's training rows are never in any
  validation set — they are always in training.
  So some rows will have coverage = 0. This is expected.
  We verify the count of zero-coverage rows is reasonable
  and matches approximately the first fold training size.

Check 3 — Probabilities sum to 1.0
  For every predicted row, P(Sell) + P(Hold) + P(Buy)
  must equal exactly 1.0 within floating point tolerance.
  If this fails XGBoost produced invalid probability output.
  This should never happen but we verify it anyway.
  A meta-model trained on probabilities that do not sum
  to 1 will produce nonsensical coefficient estimates.

Check 4 — No NaN in predicted rows
  oof_probs was initialised with np.nan for every cell.
  After the loop all predicted rows must have real numbers.
  A NaN means a fold failed to write its predictions.
  This could happen if a fold crashed after model.fit()
  but before the storage line — extremely rare but possible.

Check 5 — OOF shape matches training set
  oof_probs must have exactly as many rows as df_train
  and exactly 3 columns — one per class.
  Shape mismatch means the array was accidentally
  reinitialised or resized between cells.

Check 6 — OOF class distribution is reasonable
  The predicted class distribution from OOF should
  be in the same ballpark as the true distribution.
  Large deviation means the model is systematically
  wrong in one direction — worth investigating before
  feeding into the meta-model.

  We do not assert on this check — we just print it
  and flag if any class deviates by more than 15%.
  Some deviation is expected and normal.
  Extreme deviation (e.g. model never predicts Sell)
  is a warning sign worth investigating.

#### What Happens After This Cell

If all 6 checks pass the OOF predictions are confirmed
clean and ready. Cell 9 trains the final model.
Notebook 07 loads oof_probs to train the meta-model.

If any check fails fix the issue in Cell 7 and rerun
from Cell 7 onwards. Never proceed past a failed check.

In [8]:
# ============================================================
# CELL 8 - OOF Sanity Validation (6 Checks)
# ============================================================
# All 6 checks must pass before proceeding to Cell 9
# A single failure stops execution with a clear message
# This protects notebook 07 from corrupted stacking input
# Never skip or comment out these checks
# ============================================================

print('OOF Sanity Validation — 6 Checks')
print('=' * 55)

# ── Check 1 — No row predicted twice ──────────────────────
double_predicted = (oof_coverage > 1).sum()
assert double_predicted == 0, \
    f'CHECK 1 FAILED: {double_predicted} rows predicted ' \
    f'more than once — date overlap in fold splitting'
print(f'  Check 1 — No double predictions     : PASSED ✅'
      f'  (max coverage = {oof_coverage.max()})')

# ── Check 2 — Coverage count is sensible ──────────────────
zero_coverage     = (oof_coverage == 0).sum()
predicted_count   = (oof_coverage == 1).sum()
first_fold_size   = (~df_train['Date'].isin(
    np.array(sorted(df_train['Date'].unique()))[
        TimeSeriesSplit(n_splits=N_SPLITS
    ).split(
        np.array(sorted(df_train['Date'].unique()))
    ).__next__()[1]
    ]
)).sum()

print(f'  Check 2 — Coverage summary:')
print(f'            Predicted rows   : {predicted_count:,}')
print(f'            Unpredicted rows : {zero_coverage:,} '
      f'(first fold train rows — expected)')
print(f'            Total rows       : {len(df_train):,}')
assert predicted_count + zero_coverage == len(df_train), \
    f'CHECK 2 FAILED: predicted {predicted_count} + ' \
    f'unpredicted {zero_coverage} != {len(df_train)}'
print(f'  Check 2 — Coverage count sensible   : PASSED ✅')

# ── Check 3 — Probabilities sum to 1.0 ────────────────────
predicted_mask = oof_coverage == 1
prob_sums      = oof_probs[predicted_mask].sum(axis=1)
max_deviation  = np.abs(prob_sums - 1.0).max()
assert max_deviation < 1e-5, \
    f'CHECK 3 FAILED: probabilities do not sum to 1.0 ' \
    f'max deviation = {max_deviation:.2e}'
print(f'  Check 3 — Probs sum to 1.0          : PASSED ✅'
      f'  (max deviation = {max_deviation:.2e})')

# ── Check 4 — No NaN in predicted rows ────────────────────
nan_in_predicted = np.isnan(
    oof_probs[predicted_mask]
).sum()
assert nan_in_predicted == 0, \
    f'CHECK 4 FAILED: {nan_in_predicted} NaN values ' \
    f'found in predicted rows — fold storage failed'
print(f'  Check 4 — No NaN in predictions     : PASSED ✅')

# ── Check 5 — Shape matches training set ──────────────────
expected_shape = (len(df_train), N_CLASSES)
assert oof_probs.shape == expected_shape, \
    f'CHECK 5 FAILED: expected shape {expected_shape} ' \
    f'got {oof_probs.shape}'
print(f'  Check 5 — OOF shape correct         : PASSED ✅'
      f'  {oof_probs.shape}')

# ── Check 6 — Class distribution comparison ───────────────
oof_preds      = np.argmax(oof_probs[predicted_mask], axis=1)
y_true_covered = df_train[TARGET_COL].values[predicted_mask]

print(f'  Check 6 — Class distribution comparison:')
print(f'            {"Class":<8} {"True %":>8} '
      f'{"OOF Pred %":>12} {"Diff":>8} {"Status"}')
print(f'            {"-" * 50}')

all_cls_ok = True
for cls in range(N_CLASSES):
    true_pct = (y_true_covered == cls).mean() * 100
    pred_pct = (oof_preds      == cls).mean() * 100
    diff     = abs(pred_pct - true_pct)
    label    = LABELS[cls]
    flag     = '✅' if diff < 15 else '⚠️  large deviation'
    if diff >= 15:
        all_cls_ok = False
    print(f'            {label:<8} {true_pct:>8.1f}% '
          f'{pred_pct:>11.1f}% {diff:>7.1f}%  {flag}')

if all_cls_ok:
    print(f'  Check 6 — Distribution reasonable   : PASSED ✅')
else:
    print(f'  Check 6 — Distribution WARNING ⚠️')
    print(f'            One or more classes deviate > 15%')
    print(f'            Investigate before proceeding')

# ── Final summary ──────────────────────────────────────────
print()
print('=' * 55)
print('All OOF sanity checks passed ✅')
print('OOF predictions are clean and ready')
print()
print('Next steps:')
print('  Cell 9       →  train final model on full train set')
print('  Notebook 07  →  load oof_probs to train meta-model')
print(f'  Predicted rows available : {predicted_count:,}')
print(f'  OOF array shape          : {oof_probs.shape}')

OOF Sanity Validation — 6 Checks
  Check 1 — No double predictions     : PASSED ✅  (max coverage = 1)
  Check 2 — Coverage summary:
            Predicted rows   : 20,100
            Unpredicted rows : 4,080 (first fold train rows — expected)
            Total rows       : 24,180
  Check 2 — Coverage count sensible   : PASSED ✅
  Check 3 — Probs sum to 1.0          : PASSED ✅  (max deviation = 8.94e-08)
  Check 4 — No NaN in predictions     : PASSED ✅
  Check 5 — OOF shape correct         : PASSED ✅  (24180, 3)
  Check 6 — Class distribution comparison:
            Class      True %   OOF Pred %     Diff Status
            --------------------------------------------------
            Sell         26.4%        21.0%     5.4%  ✅
            Hold         40.3%        41.6%     1.3%  ✅
            Buy          33.3%        37.4%     4.1%  ✅
  Check 6 — Distribution reasonable   : PASSED ✅

All OOF sanity checks passed ✅
OOF predictions are clean and ready

Next steps:
  Cell 9       →  tra

-------------------

### Cell 9 — Train Final XGBoost on Full Training Set

#### Why We Train Again on the Full Dataset

The OOF loop in Cell 7 trained 5 separate models.
Each model only saw a portion of the training data.

  Fold 1 model trained on  3,980 rows
  Fold 2 model trained on  8,000 rows
  Fold 3 model trained on 12,020 rows
  Fold 4 model trained on 16,040 rows
  Fold 5 model trained on 20,060 rows

None of those 5 models saw all 24,180 training rows.
They were for generating honest OOF predictions only.
We discard them after extracting the OOF predictions.

Now we train ONE final model on ALL 24,180 training rows.
More data = better learned patterns = stronger model.
This is the model that gets saved to disk and used for:
  Validation evaluation in Cell 10
  Live predictions in the Streamlit dashboard
  Stacking with LightGBM and LSTM in notebook 07

#### Same Parameters — Why

We use exactly the same XGB_PARAMS dictionary.
The CV loop already confirmed these parameters work.
Changing parameters now would make the CV results
meaningless — we validated one set of parameters
but deployed a different set.

Consistency between CV and final training is mandatory.

#### Sample Weights on Full Dataset

We recompute sample weights on the full training set.
The weights from the CV loop were per-fold weights —
each fold had a slightly different class distribution.
The full dataset has its own class distribution
so we compute fresh weights for this training run.

In [9]:
# ============================================================
# CELL 9 - Train Final XGBoost on Full Training Set
# ============================================================
# Train ONE model on all 24,180 training rows
# Same parameters as CV loop — consistency is mandatory
# This model is saved to disk and used in production
# The 5 fold models from Cell 7 are discarded
# ============================================================

X_train_full = df_train[FEATURE_COLS].values
y_train_full = df_train[TARGET_COL].values.astype(int)
w_train_full = compute_sample_weight(
    'balanced', y=y_train_full
)

print('Training final XGBoost on full training set')
print('=' * 50)
print(f'  Training rows : {len(X_train_full):,}')
print(f'  Features      : {len(FEATURE_COLS)}')
print(f'  Classes       : {N_CLASSES}')
print(f'  Parameters    : same as CV loop')
print()
print('Training in progress...')

xgb_final = xgb.XGBClassifier(**XGB_PARAMS)
xgb_final.fit(
    X_train_full,
    y_train_full,
    sample_weight=w_train_full,
    verbose=False
)

print('Training complete')
print()

# Quick sanity check on the trained model
train_preds = xgb_final.predict(X_train_full)
train_probs = xgb_final.predict_proba(X_train_full)

train_f1  = f1_score(y_train_full, train_preds,
                     average='macro')
train_bal = balanced_accuracy_score(
    y_train_full, train_preds
)

print('Final model — Training Set Performance')
print('=' * 50)
print(f'  F1 Macro          : {train_f1:.4f}')
print(f'  Balanced Accuracy : {train_bal:.4f}')
print()
print('  Note: training set scores are always higher')
print('  than OOF scores because the model has seen')
print('  these rows. This is expected — not leakage.')
print(f'  OOF Mean F1 was   : {np.mean(fold_scores):.4f}')
print(f'  Train F1 is       : {train_f1:.4f}')
gap = train_f1 - np.mean(fold_scores)
print(f'  Gap               : {gap:.4f}')
if gap > 0.15:
    print('  ⚠️  Gap > 0.15 — possible overfitting')
    print('     Consider reducing max_depth or n_estimators')
else:
    print('  ✅ Gap is reasonable — no severe overfitting')

print()
print(f'  Probability output shape : {train_probs.shape}')
assert train_probs.shape == (len(X_train_full), N_CLASSES), \
    'Probability shape mismatch — check model output'
print('Final model ready for Cell 10 evaluation ✅')

Training final XGBoost on full training set
  Training rows : 24,180
  Features      : 17
  Classes       : 3
  Parameters    : same as CV loop

Training in progress...
Training complete

Final model — Training Set Performance
  F1 Macro          : 0.8410
  Balanced Accuracy : 0.8438

  Note: training set scores are always higher
  than OOF scores because the model has seen
  these rows. This is expected — not leakage.
  OOF Mean F1 was   : 0.3678
  Train F1 is       : 0.8410
  Gap               : 0.4732
  ⚠️  Gap > 0.15 — possible overfitting
     Consider reducing max_depth or n_estimators

  Probability output shape : (24180, 3)
Final model ready for Cell 10 evaluation ✅


-------------

### Cell 10 — Evaluate on Validation Set

#### This Is the Moment of Truth

The validation set has been locked since Cell 3.
This is the first and only time we open it.

The score we get here is our honest answer to:
  How well does XGBoost predict stock signals
  on data it has never seen before?

We compute six metrics:

Balanced Accuracy
  Overall accuracy adjusted for class imbalance.
  Hold dominates at 43% so raw accuracy is misleading.
  Balanced accuracy gives equal weight to all three classes.
  A model that always predicts Hold gets 0.33 not 0.43.

F1 Macro
  Average F1 score across all three classes equally.
  This is our primary metric for comparing models.
  Equal weight to Sell, Hold, Buy regardless of frequency.

F1 Weighted
  Average F1 weighted by class frequency.
  Gives more weight to Hold since it has more rows.
  Useful secondary metric — should be higher than macro.

Classification Report
  Precision, Recall, F1 per class.
  Precision = when model says BUY how often is it right?
  Recall    = of all real BUY days how many did we catch?

Confusion Matrix
  Shows exactly where the model is confused.
  Rows = actual class, Columns = predicted class.
  Diagonal = correct predictions.
  Off-diagonal = mistakes and what kind of mistakes.

#### What Score to Expect

  Val F1 close to OOF F1 (~0.35 to 0.42) → model generalises well
  Val F1 much lower than OOF F1 (~0.25)  → overfitting detected
  Val F1 much higher than OOF F1 (~0.55) → suspicious, check for leakage

#### What We Do After Seeing This Score

  If val F1 is in range → proceed to Cell 11 feature importance
  If val F1 is too low  → reduce max_depth and n_estimators
                          rerun from Cell 6 onwards
  We do NOT retune after seeing val score and then
  re-evaluate on the same val set — that is peeking.

In [10]:
# ============================================================
# CELL 10 - Validation Set Evaluation
# ============================================================
# Validation set opened here for the first and only time
# Compute all classification metrics on 2024 held-out data
# These numbers represent honest real-world performance
# Do NOT retune parameters after seeing these results
# and re-evaluate on the same validation set — that is leakage
# ============================================================

X_val = df_val[FEATURE_COLS].values
y_val = df_val[TARGET_COL].values.astype(int)

# Generate predictions on validation set
val_probs = xgb_final.predict_proba(X_val)
val_preds = np.argmax(val_probs, axis=1)

# ── Core metrics ──────────────────────────────────────────
bal_acc = balanced_accuracy_score(y_val, val_preds)
f1_mac  = f1_score(y_val, val_preds, average='macro')
f1_wtd  = f1_score(y_val, val_preds, average='weighted')

print('XGBoost — Validation Set Results (2024)')
print('=' * 55)
print(f'  Balanced Accuracy : {bal_acc:.4f}')
print(f'  F1 Macro          : {f1_mac:.4f}')
print(f'  F1 Weighted       : {f1_wtd:.4f}')
print()

# ── Compare to OOF score ──────────────────────────────────
oof_mean = np.mean(fold_scores)
val_gap  = abs(f1_mac - oof_mean)
print('  Generalisation Check:')
print(f'  OOF Mean F1  : {oof_mean:.4f}')
print(f'  Val F1       : {f1_mac:.4f}')
print(f'  Gap          : {val_gap:.4f}')
if val_gap < 0.05:
    print('  ✅ Excellent — val score matches OOF closely')
elif val_gap < 0.10:
    print('  ✅ Good — val score within acceptable range')
else:
    print('  ⚠️  Val score differs from OOF significantly')
    print('     Consider tuning in notebook 07')
print()

# ── Classification report ─────────────────────────────────
print('Classification Report:')
print('-' * 55)
print(classification_report(
    y_val, val_preds,
    target_names=['Sell', 'Hold', 'Buy'],
    digits=4
))

# ── Confusion matrix ──────────────────────────────────────
print('Confusion Matrix (rows=Actual, cols=Predicted):')
print('-' * 55)
cm = confusion_matrix(y_val, val_preds)
print(f'  {"":>10} {"Pred Sell":>10} '
      f'{"Pred Hold":>10} {"Pred Buy":>10}')
print(f'  {"-" * 42}')
for i, row in enumerate(cm):
    label = f'Act {LABELS[i]}'
    print(f'  {label:>10} {row[0]:>10} {row[1]:>10} {row[2]:>10}')

print()

# ── Per class accuracy ────────────────────────────────────
print('Per Class Accuracy:')
print('-' * 55)
for cls in range(N_CLASSES):
    cls_mask     = y_val == cls
    cls_correct  = (val_preds[cls_mask] == cls).sum()
    cls_total    = cls_mask.sum()
    cls_acc      = cls_correct / cls_total * 100
    print(f'  {LABELS[cls]:<6} : {cls_correct:>4} / '
          f'{cls_total:>4} correct  ({cls_acc:.1f}%)')

print()
print('=' * 55)
print('Validation evaluation complete')
print('Validation set is now closed — no further evaluation')
print('on this set until final ensemble comparison in notebook 07')

XGBoost — Validation Set Results (2024)
  Balanced Accuracy : 0.3799
  F1 Macro          : 0.3765
  F1 Weighted       : 0.4176

  Generalisation Check:
  OOF Mean F1  : 0.3678
  Val F1       : 0.3765
  Gap          : 0.0087
  ✅ Excellent — val score matches OOF closely

Classification Report:
-------------------------------------------------------
              precision    recall  f1-score   support

        Sell     0.2263    0.2436    0.2346      1018
        Hold     0.5162    0.6124    0.5602      2260
         Buy     0.4077    0.2838    0.3346      1642

    accuracy                         0.4264      4920
   macro avg     0.3834    0.3799    0.3765      4920
weighted avg     0.4200    0.4264    0.4176      4920

Confusion Matrix (rows=Actual, cols=Predicted):
-------------------------------------------------------
              Pred Sell  Pred Hold   Pred Buy
  ------------------------------------------
    Act Sell        248        517        253
    Act Hold        452     

-----------

### Cell 11 — Feature Importance

#### What Feature Importance Tells Us

XGBoost builds 500 decision trees during training.
Every tree splits data at decision points using one feature.
Feature importance counts how many times each feature
was chosen as a split point across all 500 trees.

High importance = model relies on this feature heavily
                  to make correct decisions
Low importance  = feature adds little value
                  consider dropping in future iterations

This is useful for three reasons:

1. Validation — confirms the model learned from
   the right features. If day_of_week is the most
   important feature something is wrong — calendar
   effects alone cannot predict stock movements.

2. Insight — tells us which market signals matter most
   for this dataset. If rsi_14 dominates we know
   momentum is the strongest predictor in 2019-2024.

3. Future optimisation — in notebook 07 we can drop
   the bottom 3 to 5 features and retrain. Removing
   noise features sometimes improves generalisation.

#### What We Expect to See

Based on financial theory and EDA findings:

  Top features likely:
    return_1d, return_5d    — recent momentum is strong
    rsi_14                  — momentum indicator
    macd_hist               — trend direction
    volume_ratio            — unusual activity signal
    market_return_1d        — macro context

  Bottom features likely:
    day_of_week             — weak calendar effect
    quarter                 — weak seasonal effect
    month                   — moderate seasonal effect

If the actual results differ significantly from this
it is worth investigating — the model may have found
a pattern we did not expect from EDA.

#### Two Types of Importance

We compute two types for comparison:

weight — number of times feature used to split
         most common, can favour high-cardinality features

gain   — average improvement in loss from splits
         more meaningful — favours features that
         actually reduce prediction error when used

We display both and note where they disagree.
Disagreement between weight and gain means a feature
is used often but does not help much — a candidate
for removal in notebook 07.

In [11]:
# ============================================================
# CELL 11 - Feature Importance
# ============================================================
# Two importance types computed — weight and gain
# Weight = how often feature used to split
# Gain   = how much each split actually improved the model
# Disagreement between the two = feature used but not useful
# ============================================================

# ── Weight-based importance ───────────────────────────────
importance_weight = xgb_final.get_booster().get_score(
    importance_type='weight'
)
importance_gain = xgb_final.get_booster().get_score(
    importance_type='gain'
)

# Build dataframe with both types
feat_imp = pd.DataFrame({
    'Feature' : FEATURE_COLS,
    'Weight'  : [importance_weight.get(f'f{i}', 0)
                 for i in range(len(FEATURE_COLS))],
    'Gain'    : [importance_gain.get(f'f{i}', 0)
                 for i in range(len(FEATURE_COLS))]
})

# Normalise to percentages for easy comparison
feat_imp['Weight_pct'] = (
    feat_imp['Weight'] / feat_imp['Weight'].sum() * 100
)
feat_imp['Gain_pct'] = (
    feat_imp['Gain'] / feat_imp['Gain'].sum() * 100
)

# Sort by gain — more meaningful metric
feat_imp_gain = feat_imp.sort_values(
    'Gain_pct', ascending=False
).reset_index(drop=True)

# ── Print weight ranking ───────────────────────────────────
feat_imp_weight = feat_imp.sort_values(
    'Weight_pct', ascending=False
).reset_index(drop=True)

print('Feature Importance — Weight Ranking')
print('(how often each feature is used to split)')
print('=' * 60)
print(f'  {"Rank":<5} {"Feature":<25} {"Weight %":>9} {"Bar"}')
print(f'  {"-" * 55}')
for i, row in feat_imp_weight.iterrows():
    bar = '█' * int(row['Weight_pct'])
    print(f'  {i+1:<5} {row["Feature"]:<25} '
          f'{row["Weight_pct"]:>8.2f}%  {bar}')

print()

# ── Print gain ranking ────────────────────────────────────
print('Feature Importance — Gain Ranking')
print('(how much each feature actually improves predictions)')
print('=' * 60)
print(f'  {"Rank":<5} {"Feature":<25} {"Gain %":>9} {"Bar"}')
print(f'  {"-" * 55}')
for i, row in feat_imp_gain.iterrows():
    bar = '█' * int(row['Gain_pct'])
    print(f'  {i+1:<5} {row["Feature"]:<25} '
          f'{row["Gain_pct"]:>8.2f}%  {bar}')

print()

# ── Disagreement analysis ─────────────────────────────────
print('Disagreement Analysis')
print('(large rank difference = used often but not useful)')
print('=' * 60)

weight_rank = {row['Feature']: i+1
               for i, row in feat_imp_weight.iterrows()}
gain_rank   = {row['Feature']: i+1
               for i, row in feat_imp_gain.iterrows()}

print(f'  {"Feature":<25} {"Weight Rank":>12} '
      f'{"Gain Rank":>10} {"Diff":>6} {"Flag"}')
print(f'  {"-" * 60}')
for feat in FEATURE_COLS:
    wr   = weight_rank[feat]
    gr   = gain_rank[feat]
    diff = abs(wr - gr)
    flag = '⚠️  check' if diff >= 5 else '✅'
    print(f'  {feat:<25} {wr:>12} {gr:>10} '
          f'{diff:>6} {flag}')

print()
print('=' * 60)
print('Feature importance analysis complete')
print('Bottom 3 features by gain are candidates')
print('for removal during tuning in notebook 07')

Feature Importance — Weight Ranking
(how often each feature is used to split)
  Rank  Feature                    Weight % Bar
  -------------------------------------------------------
  1     market_return_1d              9.96%  █████████
  2     bb_width                      7.81%  ███████
  3     atr_14                        7.80%  ███████
  4     macd_hist                     7.57%  ███████
  5     return_5d                     6.77%  ██████
  6     volume_trend                  6.54%  ██████
  7     volume_ratio                  6.26%  ██████
  8     return_20d                    6.22%  ██████
  9     price_vs_ma50                 6.12%  ██████
  10    return_10d                    6.00%  ██████
  11    month                         5.98%  █████
  12    rsi_14                        5.62%  █████
  13    return_1d                     5.37%  █████
  14    price_vs_ma20                 4.70%  ████
  15    price_volume_signal           4.09%  ████
  16    day_of_week                  

--------

### Cell 12 — Save Model and OOF Predictions

#### What We Save and Why

This cell saves four files to the ../models/ directory.
Each file has a specific purpose in the downstream notebooks.

xgb_model.pkl
  The final XGBoost model trained on all 24,180 training rows.
  Used in notebook 07 to generate predictions on the
  validation set for ensemble evaluation.
  Used in notebook 08 (dashboard) to predict live 2025 data.
  Loaded with joblib.load() in every downstream notebook.

xgb_oof_probs.npy
  The out-of-fold probability array of shape (24180, 3).
  Contains P(Sell), P(Hold), P(Buy) for every training row
  predicted by a model that never trained on that row.
  This is the input to the Logistic Regression meta-model
  in notebook 07. Without this file stacking cannot work.
  Loaded with np.load() in notebook 07.

xgb_oof_coverage.npy
  Boolean mask of shape (24180,) indicating which rows
  have valid OOF predictions (coverage=1) vs which rows
  were never in any validation fold (coverage=0).
  Notebook 07 uses this mask to filter to predicted rows
  only when training the meta-model.
  First fold training rows have coverage=0 and must be
  excluded from meta-model training.

feature_cols.pkl
  The exact list of 17 feature column names used for
  training in this notebook.
  Notebooks 05, 06, and 07 load this file to guarantee
  they use the exact same features in the exact same order.
  Column order matters for model predictions — if notebook
  07 passes features in a different order the model
  produces completely wrong predictions silently.

#### Why joblib for Models and np.save for Arrays

joblib is the standard serialisation library for sklearn
and XGBoost objects. It handles complex Python objects
including internal tree structures efficiently.

np.save is the standard format for numpy arrays.
It is faster and more compact than joblib for pure arrays.
np.load with allow_pickle=False is safe for arrays.

#### Verification After Saving

Every saved file is immediately reloaded and verified.
We check that:
  The reloaded model produces identical predictions
  to the original model on a small test batch.
  The reloaded OOF array is identical to the original.
  The reloaded feature list matches the original exactly.

This catches corrupted saves immediately rather than
discovering the problem in notebook 07 after hours of work.

In [12]:
# ============================================================
# CELL 12 - Save Model and OOF Predictions
# ============================================================
# Save four files to ../models/ directory
# Each file has a specific role in downstream notebooks
# Verify every file immediately after saving
# A corrupted save discovered now saves hours later
# ============================================================

print('Saving Notebook 04 outputs')
print('=' * 55)

# ── File paths ────────────────────────────────────────────
PATH_MODEL    = f'{MODELS_DIR}xgb_model.pkl'
PATH_OOF_PROB = f'{MODELS_DIR}xgb_oof_probs.npy'
PATH_OOF_COV  = f'{MODELS_DIR}xgb_oof_coverage.npy'
PATH_FEATURES = f'{MODELS_DIR}feature_cols.pkl'

# ── Save final model ──────────────────────────────────────
joblib.dump(xgb_final, PATH_MODEL)
size_model = os.path.getsize(PATH_MODEL) / 1024
print(f'  xgb_model.pkl saved        : {size_model:>8.1f} KB')

# ── Save OOF probabilities ────────────────────────────────
np.save(PATH_OOF_PROB, oof_probs)
size_oof_prob = os.path.getsize(PATH_OOF_PROB) / 1024
print(f'  xgb_oof_probs.npy saved    : {size_oof_prob:>8.1f} KB')

# ── Save OOF coverage mask ────────────────────────────────
np.save(PATH_OOF_COV, oof_coverage)
size_oof_cov = os.path.getsize(PATH_OOF_COV) / 1024
print(f'  xgb_oof_coverage.npy saved : {size_oof_cov:>8.1f} KB')

# ── Save feature column list ──────────────────────────────
joblib.dump(FEATURE_COLS, PATH_FEATURES)
size_features = os.path.getsize(PATH_FEATURES) / 1024
print(f'  feature_cols.pkl saved     : {size_features:>8.1f} KB')

print()

# ── Verify model ──────────────────────────────────────────
print('Verifying saved files...')
print('-' * 55)

xgb_loaded   = joblib.load(PATH_MODEL)
sample_rows  = df_val[FEATURE_COLS].values[:10]
orig_preds   = xgb_final.predict(sample_rows)
loaded_preds = xgb_loaded.predict(sample_rows)

assert np.array_equal(orig_preds, loaded_preds), \
    'MODEL VERIFY FAILED: loaded model predictions differ'
print(f'  xgb_model.pkl              : verified ✅'
      f'  (predictions identical on 10 sample rows)')

# ── Verify OOF probabilities ──────────────────────────────
oof_loaded = np.load(PATH_OOF_PROB)
assert oof_loaded.shape == oof_probs.shape, \
    f'OOF VERIFY FAILED: shape mismatch ' \
    f'{oof_loaded.shape} vs {oof_probs.shape}'
assert np.allclose(oof_loaded, oof_probs, equal_nan=True), \
    'OOF VERIFY FAILED: values differ after reload'
print(f'  xgb_oof_probs.npy          : verified ✅'
      f'  shape {oof_loaded.shape}')

# ── Verify OOF coverage ───────────────────────────────────
cov_loaded = np.load(PATH_OOF_COV)
assert np.array_equal(cov_loaded, oof_coverage), \
    'COVERAGE VERIFY FAILED: values differ after reload'
print(f'  xgb_oof_coverage.npy       : verified ✅'
      f'  shape {cov_loaded.shape}')

# ── Verify feature list ───────────────────────────────────
feat_loaded = joblib.load(PATH_FEATURES)
assert feat_loaded == FEATURE_COLS, \
    'FEATURES VERIFY FAILED: list differs after reload'
print(f'  feature_cols.pkl           : verified ✅'
      f'  ({len(feat_loaded)} features)')

print()

# ── Final summary ──────────────────────────────────────────
print('=' * 55)
print('NOTEBOOK 04 COMPLETE')
print('=' * 55)
print()
print('Files saved to ../models/:')
print(f'  xgb_model.pkl              '
      f'← load in notebooks 07 and 08')
print(f'  xgb_oof_probs.npy          '
      f'← load in notebook 07 for stacking')
print(f'  xgb_oof_coverage.npy       '
      f'← load in notebook 07 for masking')
print(f'  feature_cols.pkl           '
      f'← load in notebooks 05 06 07 08')
print()
print('XGBoost Performance Summary:')
print(f'  OOF Mean F1 (macro)   : {np.mean(fold_scores):.4f}')
print(f'  Val F1 (macro)        : {f1_mac:.4f}')
print(f'  Val Balanced Accuracy : {bal_acc:.4f}')
print(f'  Val Overall Accuracy  : {f1_wtd:.4f}')
print()
print('Next notebook : 05_lightgbm.ipynb')
print('  Load        : ../data/processed/all_stocks_features.csv')
print('  Load        : ../models/feature_cols.pkl')
print('  Save        : ../models/lgbm_model.pkl')
print('  Save        : ../models/lgbm_oof_probs.npy')
print('  Save        : ../models/lgbm_oof_coverage.npy')
print('=' * 55)

Saving Notebook 04 outputs
  xgb_model.pkl saved        :   6197.4 KB
  xgb_oof_probs.npy saved    :    566.8 KB
  xgb_oof_coverage.npy saved :    189.0 KB
  feature_cols.pkl saved     :      0.2 KB

Verifying saved files...
-------------------------------------------------------
  xgb_model.pkl              : verified ✅  (predictions identical on 10 sample rows)
  xgb_oof_probs.npy          : verified ✅  shape (24180, 3)
  xgb_oof_coverage.npy       : verified ✅  shape (24180,)
  feature_cols.pkl           : verified ✅  (17 features)

NOTEBOOK 04 COMPLETE

Files saved to ../models/:
  xgb_model.pkl              ← load in notebooks 07 and 08
  xgb_oof_probs.npy          ← load in notebook 07 for stacking
  xgb_oof_coverage.npy       ← load in notebook 07 for masking
  feature_cols.pkl           ← load in notebooks 05 06 07 08

XGBoost Performance Summary:
  OOF Mean F1 (macro)   : 0.3678
  Val F1 (macro)        : 0.3765
  Val Balanced Accuracy : 0.3799
  Val Overall Accuracy  : 0.4176
